# The MongoDB Data Model
The MongoDB Data Model uses a referenced structure. Three seperate collections, those being vehicle, camera and violations, store the data independently.

This type of data model has multiple benefits such as write efficiency, consistency and normalization. Splitting the data into distinct domain entities simplifies write operations by using unique identifiers to reference related data. Additionally, compared to an embedded model, it reduces data repetition and improves consistency and data integrity is ensured with the referenced layout.

However, this model has operaton tradeoffs compared to the embedded model. When making queries that require looking up data from different collections, the system is required to run memory-intensive aggregation lookups, which introduces network overhead, raises query latency and spikes server memory utilization under high read volumes

### vehicle collection table of fields
| Field Name | Data Type | Key Type | Meaning |
| ----- | ----- | ----- | ----- |
| _id | ObjectID | Primary | Unique document identifier by MongoDB |
| car_plate | String | Unique | Vehicle's car plate |
| owner_name | String | - | Name of vehicle's owner |
| owner_addr | String | - | Address of vehicle's owner |
| vehicle_type | String | - | Type of vehicle |
| registration_date | String/Date | - | Date/time of vehicle's registration |

### camera collection table of fields
| Field Name | Data Type | Key Type | Meaning |
| ----- | ----- | ----- | ----- |
| _id | ObjectID | Primary | Unique document identifier by MongoDB |
| latitude | Double/Float | - | The camera's geographical latitude |
| longitude | Double/Float | - | The camera's geographical longitude |
| position | Double/Float | - | The physical location marker of the camera on the road in km |
| speed_limit | Integer | - | The camera's legal maximum speed in km/h |

### violations collection table of fields
| Field Name | Data Type | Key Type | Meaning |
| ----- | ----- | ----- | ----- |
| _id | ObjectID | Primary | Unique document identifier by MongoDB |
| violation_id | String | Unique | Unique tracking identifier for the speeding incident |
| car_plate | String | Foreign | The speeding vehicle's car plate (links to car_plate field in the vehicle collection) |
| camera_id_start | Integer | Foreign | The ID of the entry camera (links to camera_id field in the camera collection) |
| camera_id_end | Integer | Foreign | The ID of the exit camera (links to camera_id field in the camera collection) |
| timestamp_start | String/Date | - | Date/time when the entry camera detected the vehicle |
| timestamp_end | String/Date | - | Date/time when the exit camera detected the vehicle |
| speed_reading | Double/Float | - | Calculated speed of the vehicle between the entry and exit cameras in km/h |

![Alt Text](/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/visuals/mongodb_workflow_diagram.png)

First, import the the pymongo and pandas libraries. From pymongo specifically you would need to import the MongoClient class.

In [9]:
from pymongo import MongoClient
import pandas as pd

Local directory locations for the CSV files of raw camera, vehicle, camera event and historic violation data are specified and the IP address is configured to define the host IP address where the target MongoDB instance is running.

In [10]:
camera_file_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera.csv"
vehicle_file_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/vehicle.csv"
violations_file_path = "/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/data/camera_event_historic.csv"
ip_address = '192.168.1.108'

The client is set up to establish a connection to MongoDB using the IP address and the port (27017).

In [11]:
client = MongoClient(ip_address, 27017)
db = client.a2_db

Import the data from the CSV files for camera, vehicle and violations data, these would later be used as data to store in the MongoDB collections later.

In [12]:
# Extract the data from the CSV files and convert them into pandas dataframes
camera_data = pd.read_csv(camera_file_path)
vehicle_data = pd.read_csv(vehicle_file_path)
violations_data = pd.read_csv(violations_file_path)

# Align camera schema and remove indexing remnants
camera_data.set_index(camera_data.camera_id, inplace = True)
camera_data.drop("camera_id", axis = 1, inplace = True)

Set up the collections in the client.

In [13]:
camera = db.camera
vehicle = db.vehicle
violations = db.violations

Drop the collections. This ensures a fresh environment setup.

In [14]:
camera.drop()
vehicle.drop()
violations.drop()

Convert the pandas dataframes into dictionaries.

In [15]:
# The below code is derived from zero (23 April 2015) and wjandrea. (8 June 2025) https://stackoverflow.com/questions/29815129/pandas-dataframe-to-list-of-dictionaries
camera_as_dict = camera_data.to_dict('records')
vehicle_as_dict = vehicle_data.to_dict('records')
violations_as_dict = violations_data.to_dict('records')

Insert all data from the dictionaries into the collections.

In [16]:
camera.insert_many(camera_as_dict)
vehicle.insert_many(vehicle_as_dict)
violations.insert_many(violations_as_dict)
